# Unitree-Go2-Flat-MethodA-Electric — 계산 흐름 노트

본 노트의 목적은 한 정책 주기 안에서 q, q̇, τ, I 가 어떻게 전파되는지를
연구실 구성원이 **값을 직접 넣어가며** 따라가는 데 있다.

각 단계는 다음 3박자 셀로 구성된다.
1. **(A) 소스 발췌** — 프로젝트 코드를 그대로 인용 (`# 출처: <파일>:<라인>`)
2. **(B) 값 대입** — 동일한 식을 1∼2 자유도 축소 예제로 옮겨 직접 계산
3. **(C) 중간 결과** — 다음 절로 넘어가는 산출물 한 줄 요약

추측은 쓰지 않는다. 코드 본문을 못 본 부분은 8절 미확인 항목에 모은다.


## 0. 개요

- 대상 시스템: Unitree Go2 (12 관절 사족보행 로봇), 평지 보행 task
- task ID: `Unitree-Go2-Flat-MethodA-Electric`
- 시간 척도 (코드 확인 결과):
    - 정책 주기:    **20 ms**
    - PD 재계산 주기: **5 ms**
    - 물리 적분 주기: **0.1 ms**
- "MethodA" 의 의미 (코드 확인): BE 일관 — 적분기 / Schur / Force RHS 모두
  $\beta = 1/(1 + h/\tau)$. patched mjwarp 의 `dynprm[4] = 0`.
  - 기호: $\beta$ 는 1-step 필터/implicit 적분 계수, $h$ 는 물리 적분 시간 간격,
    $\tau$ 는 필터 시정수다.
- "Electric" 의 의미 (코드 확인): 모터 전류 $I$ 를 MuJoCo 의 activation
  state (`d->act`) 에 통합한 BLDC 모터 모델 (`dyntype = filterexact` +
  Schur cross-Jacobian).

> NOTE, hyunnnchoi, 2026.05.02 — ZOH 와 `filterexact` 용어를 먼저 정의한다.
>
> - **ZOH (zero-order hold)**: 어떤 시점에 계산한 값을 다음 갱신 시점까지 상수로
>   유지하는 sample-and-hold 방식이다. 예를 들어 $u_k$ 를 $t_k$ 에 계산하면
>   $t \in [t_k, t_{k+1})$ 동안 $u(t) = u_k$ 로 쓴다.
>   기호: $u_k$ 는 $k$번째 갱신 시점에서 계산된 값, $t_k$ 와 $t_{k+1}$ 은
>   연속한 두 갱신 시각, $u(t)$ 는 실제 시간 $t$ 에 actuator/제어기가 참조하는 값이다.
> - **`filterexact`**: MuJoCo actuator dynamics 타입 중 하나로, activation state 를
>   1차 필터처럼 적분한다. 이 노트에서는 activation 을 전류 $I$ 로 해석하고,
>   `d->ctrl` 로 들어온 목표 전류와 현재 전류의 차이를 `act_dot` 으로 만든 뒤
>   `act = ...` 식으로 다음 전류를 계산한다.

> 시간 척도 출처:
> `src/tasks/velocity/config/go2/env_cfgs.py:189-219` 와
> `src/assets/robots/unitree_go2/go2_constants.py:305-306`
> (`_COUPLED_SUBSTEPS = 200`, `_PD_RECOMPUTE = 50`).


## 1. 시간 척도 계층

> NOTE, hyunnnchoi, 2026.05.02 — 유지되는 값은 약어 대신 의미를 풀어 쓴다.

```mermaid
flowchart TB
    P["정책 (20 ms, 1회)"] -- "q_des(t)=q_des,k" --> PD1["PD #1 (5 ms)"]
    P -- "q_des(t)=q_des,k" --> PD2["PD #2"]
    P -- "q_des(t)=q_des,k" --> PD3["PD #3"]
    P -- "q_des(t)=q_des,k" --> PD4["PD #4"]
    PD1 -- "τ_des, I_des: 50 substep 동안 cache 값 유지" --> PHY1["물리 0.1 ms × 50"]
    PD2 -- "τ_des, I_des" --> PHY2["물리 × 50"]
    PD3 -- "τ_des, I_des" --> PHY3["물리 × 50"]
    PD4 -- "τ_des, I_des" --> PHY4["물리 × 50"]
    PHY1 -- "q, q̇, I" --> PD2
    PHY2 -- "q, q̇, I" --> PD3
    PHY3 -- "q, q̇, I" --> PD4
    PHY4 -- "q, q̇, I" --> P
```

화살표에 적힌 것이 단계 사이에 전달되는 물리량이다.


## 2. 정책 단계 (20 ms)

### (A) 소스 발췌 — task 등록과 시간 설정

```python
# 출처: src/tasks/velocity/config/go2/__init__.py:81-93
register_mjlab_task(
  task_id="Unitree-Go2-Flat-MethodA-Electric",
  env_cfg=unitree_go2_flat_methoda_electric_env_cfg(
    use_velocity_action=_methoda_use_vel,
  ),
  play_env_cfg=unitree_go2_flat_methoda_electric_env_cfg(
    play=True, use_velocity_action=_methoda_use_vel,
  ),
  rl_cfg=unitree_go2_methoda_electric_ppo_runner_cfg(
    action_type=_METHODA_ACTION_TYPE,
  ),
  runner_cls=VelocityOnPolicyRunner,
)
```

```python
# 출처: src/tasks/velocity/config/go2/env_cfgs.py:189-219
def unitree_go2_flat_methoda_electric_env_cfg(
  play: bool = False,
  use_velocity_action: bool = False,
) -> ManagerBasedRlEnvCfg:
  cfg = unitree_go2_flat_env_cfg(play=play)
  cfg.scene.entities = {"robot": get_go2_methoda_robot_cfg()}
  cfg.sim.mujoco.timestep = 0.0001   # 0.1ms
  cfg.decimation = 200                # 0.1ms × 200 = 20ms policy dt
  ...
  return cfg
```

→ 정책 주기 = `timestep × decimation` = 0.1 ms × 200 = **20 ms**.


### (B) 값 대입 — 관측과 정책 출력 q_des

> NOTE, hyunnnchoi, 2026.05.02 — 정책 출력 수식과 아래 코드의 대응을 명시한다.

정책 단계에서 계산되는 목표 관절각을 다음 수식으로 둔다.

$$
q_{\mathrm{des},k} = \pi_\theta(o_k)
$$

기호:
- $q_{\mathrm{des},k}$: $k$번째 정책 step 에서 나온 목표 관절각 벡터
- $\pi_\theta$: 파라미터 $\theta$ 를 가진 정책 신경망
- $o_k$: $k$번째 정책 step 의 관측 벡터

아래 코드는 신경망 $\pi_\theta$ 자체를 실행하지 않고, 이 수식의 결과값
$q_{\mathrm{des},k}$ 를 `q_des` 배열로 직접 지정해 3 관절 축소 예제에 넣는다.
12 관절 중 앞다리 우측 (`FR_hip`, `FR_thigh`, `FR_calf`) 3 개만 사용.

In [ ]:
import numpy as np

# 관측 (이 값을 바꿔보세요)
v_body     = np.array([0.5, 0.0, 0.0])    # 동체 선속도   [m/s]
omega_body = np.array([0.0, 0.0, 0.0])    # 동체 각속도   [rad/s]
g_proj     = np.array([0.0, 0.0, -1.0])   # 중력 투영 (정상 자세)
v_cmd      = np.array([0.5, 0.0, 0.0])    # 속도 명령

# 직전 관절 상태 (FR_hip, FR_thigh, FR_calf 순)
q          = np.array([ 0.00,  0.90, -1.80])  # [rad]
q_dot      = np.array([ 0.00,  0.00,  0.00])  # [rad/s]
q_des_prev = np.array([ 0.00,  0.90, -1.80])  # [rad]

# 정책이 출력했다고 가정하는 q_des (이 값을 바꿔보세요)
q_des = np.array([ 0.05,  0.92, -1.78])       # [rad]

print("관측 요약:")
print(f"  v_body = {v_body}")
print(f"  v_cmd  = {v_cmd}")
print(f"  q      = {q}")
print(f"  q_dot  = {q_dot}")
print()
print(f"정책 출력 q_des = {q_des}")


### (C) 중간 결과

> NOTE, hyunnnchoi, 2026.05.02 — ZOH 약어 대신 유지되는 수식과 코드 구현 방식을 적는다.

정책이 한 번 계산한 $q_{\mathrm{des},k}$ 는 다음 정책 갱신 전까지 상수로 유지된다.
수식으로 쓰면, 정책 시각 $t_k$ 이후 한 정책 주기 동안

$$
q_{\mathrm{des}}(t) = q_{\mathrm{des},k},
\qquad t \in [t_k, t_k + 20\,\mathrm{ms})
$$

기호:
- $q_{\mathrm{des}}(t)$: 실제 시간 $t$ 에 PD 제어기가 참조하는 목표 관절각
- $q_{\mathrm{des},k}$: 정책 시각 $t_k$ 에 한 번 계산된 목표 관절각
- $t_k$: $k$번째 정책 갱신 시각
- $[t_k, t_k + 20\,\mathrm{ms})$: 다음 정책 갱신 전까지의 20 ms 구간

이다. 위 수식은 아래 코드에서 `q_des = np.array([...])` 로 한 번 값을 정하고,
뒤의 네 번 PD 계산이 같은 `q_des` 값을 참조하는 방식으로 구현했다.


## 3. PD 제어 단계 (5 ms)

### (A) 소스 발췌 — actuator cfg + 값 유지 분기 + 토크 계산

```python
# 출처: src/assets/robots/unitree_go2/go2_constants.py:342-358
# Method A (BE consistent: integrator/Schur/Force 전부 β_be = 1/(1+h/τ)).
_MA_MOTOR = dict(Kt=0.128, Ke=0.128, R=0.3, L=1e-4, gear_ratio=6.33,
                 substeps=_COUPLED_SUBSTEPS, pd_substeps=_PD_RECOMPUTE,
                 use_coupled=True, method="A")
GO2_METHODA_HIP = NativeElectricActuatorCfg(
  target_names_expr=(".*hip_.*",), stiffness=20.0, damping=1.0,
  effort_limit=23.5, saturation_effort=23.5, velocity_limit=30.0, armature=0.01, **_MA_MOTOR)
GO2_METHODA_THIGH = NativeElectricActuatorCfg(
  target_names_expr=(".*thigh_.*",), stiffness=20.0, damping=1.0,
  effort_limit=23.5, saturation_effort=23.5, velocity_limit=30.0, armature=0.01, **_MA_MOTOR)
GO2_METHODA_CALF = NativeElectricActuatorCfg(
  target_names_expr=(".*calf_.*",), stiffness=40.0, damping=2.0,
  effort_limit=45.0, saturation_effort=45.0, velocity_limit=30.0, armature=0.02, **_MA_MOTOR)
```

```python
# 출처: src/assets/robots/unitree_go2/mj_native_electric_actuator.py:497-540
pd_period = cfg.pd_substeps if cfg.pd_substeps > 0 else cfg.substeps
recompute_pd = (cfg.substeps <= 1
                or self._sub_idx == 0
                or (cfg.pd_substeps > 0 and self._sub_idx % pd_period == 0))

if recompute_pd:
    # PD + DC motor saturation → τ_des
    tau_des = super().compute(cmd)
    ...
    I_des = tau_des / self._Ktgr

    # 캐시 저장
    if cfg.substeps > 1:
        self._I_des_hold = I_des
        self._tau_des_hold = tau_des
else:
    # PD 주기 사이: 캐시된 I_des 사용 (ZOH)
    I_des = self._I_des_hold
    tau_des = self._tau_des_hold
```

코드상 명칭 ↔ 본문 기호:
- `tau_des` ↔ τ_des
- `self._Ktgr` ↔ Kt · gr
- `cfg.pd_substeps = 50` ↔ PD 재계산 주기 = 50 × 0.1 ms = **5 ms**

> NOTE, hyunnnchoi, 2026.05.02 — PD 재계산 사이의 cache 동작을 수식으로 풀어 쓴다.

PD 재계산 시각을 $t_m$ 이라고 하면, 아래 소스는 다음 수식을 구현한다.

$$
(\tau_{\mathrm{des}}, I_{\mathrm{des}})(t) =
\begin{cases}
\mathrm{compute\_pd\_and\_motor}(q_{\mathrm{des}}, q, \dot q), & t = t_m \\
(\tau_{\mathrm{des}}, I_{\mathrm{des}})(t_m), & t_m < t < t_{m+1}
\end{cases}
$$

기호:
- $\tau_{\mathrm{des}}$: PD와 motor saturation 이후의 목표 토크
- $I_{\mathrm{des}}$: 목표 토크를 $K_t g_r$ 로 나눈 목표 전류
- $t$: 현재 물리 substep 시각
- $t_m$, $t_{m+1}$: 연속한 두 PD 재계산 시각
- $q_{\mathrm{des}}$, $q$, $\dot q$: 목표 관절각, 현재 관절각, 현재 관절속도
- $\mathrm{compute\_pd\_and\_motor}(\cdot)$: `super().compute(cmd)` 로 대표되는 PD 및 motor saturation 계산

수식의 첫 줄은 `recompute_pd` 분기에서 `super().compute(cmd)` 와
`I_des = tau_des / self._Ktgr` 로 계산하고, 둘째 줄은 `_tau_des_hold`,
`_I_des_hold` 에 저장한 값을 다시 읽는 방식으로 구현되어 있다.

`super().compute(cmd)` 본문 (PD + DC 모터 속도 saturation) 은
외부 패키지 `mjlab.actuator.dc_actuator.DcMotorActuator` 안에 있어
코드 본문을 노트북에 옮길 수 없다 → **8절 미확인**.
아래 (B) 셀의 식은 cfg 값과 actuator wrapper 의 docstring
(`mj_native_electric_actuator.py:1-46, 466-490`) 으로부터 재구성한 형태다.


### (B) 값 대입 — 1관절 (FR_hip) 축소 예제

> NOTE, hyunnnchoi, 2026.05.02 — PD/전류 변환 수식을 먼저 보이고 아래 코드로 계산한다.
>
> NOTE, hyunnnchoi, 2026.05.02 — velocity-dependent saturation 본문은 미확인이라 아래 saturation 식은 cfg/docstring 기반의 임의 재구성 예제다.

이 절의 값 대입 코드는 다음 수식을 순서대로 구현한다.

$$
\tau_{\mathrm{pd}} = K_p(q_{\mathrm{des}} - q) - K_d\dot q
$$

$$
\tau_{\max}(\omega) = \tau_{\mathrm{sat}}\max\left(0, 1 - \frac{|\omega|}{\omega_{\max}}\right)
$$

$$
\tau_{\mathrm{des}} = \mathrm{clip}\left(\tau_{\mathrm{pd}},
-\min(\tau_{\mathrm{limit}}, \tau_{\max}),
\min(\tau_{\mathrm{limit}}, \tau_{\max})\right),
\qquad
I_{\mathrm{des}} = \frac{\tau_{\mathrm{des}}}{K_t g_r}
$$

기호:
- $\tau_{\mathrm{pd}}$: stiffness/damping 으로 계산한 raw PD 토크
- $K_p$, $K_d$: 관절별 stiffness, damping gain
- $q_{\mathrm{des}}$, $q$, $\dot q$: 목표 관절각, 현재 관절각, 현재 관절속도
- $\tau_{\max}(\omega)$: 현재 관절속도 $\omega$ 에서 허용되는 motor torque-speed 한계
- $\tau_{\mathrm{sat}}$: 정지 또는 저속에서의 saturation torque
- $\omega_{\max}$: velocity limit, $\tau_{\max}$ 가 0 이 되는 속도 기준
- $\tau_{\mathrm{limit}}$: actuator effort limit
- $\mathrm{clip}(x, a, b)$: 값 $x$ 를 구간 $[a,b]$ 안으로 제한하는 함수
- $I_{\mathrm{des}}$: 목표 전류, $K_t$: torque constant, $g_r$: gear ratio

아래 코드의 `tau_pd`, `tau_max_eff`, `tau_des`, `I_des` 가 각각 위 수식의 항에 대응한다.

In [ ]:
# FR_hip (이 값을 바꿔보세요)
Kp                = 20.0     # [N·m/rad]   = stiffness
Kd                = 1.0      # [N·m·s/rad] = damping
effort_limit      = 23.5     # [N·m]
saturation_effort = 23.5     # [N·m]
velocity_limit    = 30.0     # [rad/s]

q_des_j  = 0.05    # [rad]   from 정책
q_j      = 0.00    # [rad]   직전 q
q_dot_j  = 0.00    # [rad/s] 직전 q̇   (이 값을 바꿔보세요)

# (1) PD
tau_pd = Kp * (q_des_j - q_j) - Kd * q_dot_j
print("주의: saturation 함수 형태는 미확인, 아래 값은 임의 재구성 예제입니다.")
print(f"τ_pd = {tau_pd:.4f} [N·m]")

# (2) DC motor velocity-dependent saturation
# NOTE, hyunnnchoi, 2026.05.02 — saturation 본문은 미확인이라 이 식은 임의 재구성 예제다.
#     τ_max(ω) = saturation_effort · max(0, 1 − |ω|/velocity_limit)
tau_max_eff = saturation_effort * max(0.0, 1.0 - abs(q_dot_j) / velocity_limit)

# (3) 최종 클립 (effort_limit 와 속도 saturation 중 더 작은 쪽)
tau_clip = min(effort_limit, tau_max_eff)
tau_des  = max(-tau_clip, min(tau_clip, tau_pd))
print(f"τ_max(ω) = {tau_max_eff:.4f}  →  τ_des = {tau_des:.4f} [N·m]")

# (4) 토크 → 목표 전류
Kt, gr  = 0.128, 6.33
Kt_gr   = Kt * gr
I_des   = tau_des / Kt_gr
print(f"I_des = τ_des / (Kt·gr) = {I_des:.4f} [A]   (Kt·gr = {Kt_gr:.4f})")


### (C) 중간 결과

> NOTE, hyunnnchoi, 2026.05.02 — 유지되는 값을 수식과 코드 cache 이름으로 표현한다.

PD 단계에서 계산한 값은 다음 50 개의 물리 substep 동안

$$
\tau_{\mathrm{des}}^{(r)} = \tau_{\mathrm{des}}(t_m),
\qquad
I_{\mathrm{des}}^{(r)} = I_{\mathrm{des}}(t_m),
\qquad r = 0, \dots, 49
$$

기호:
- $r$: 하나의 PD 주기 안에서의 물리 substep index
- $\tau_{\mathrm{des}}^{(r)}$, $I_{\mathrm{des}}^{(r)}$: $r$번째 물리 substep 에서 사용하는 목표 토크와 목표 전류
- $t_m$: 해당 PD 주기가 시작될 때의 PD 재계산 시각
- $0,\dots,49$: 5 ms PD 주기 안의 50개 물리 substep

로 유지된다. 위 수식은 코드에서 `_tau_des_hold`, `_I_des_hold` 에 저장한 값을
각 substep 에서 다시 읽는 방식으로 구현되어 있다. 물리 단계는 매 substep 마다
최신 $\omega$ 로 가상 전압 $V$ 만 갱신한다
(`mj_native_electric_actuator.py:542-557`).


## 4. 물리 단계 (0.1 ms)

한 substep 안에서 일어나는 계산을 4 개 절로 분해한다.


### 4.1 운동방정식

#### (A) 소스 발췌 — actuator force = Kt·gr·I

```python
# 출처: src/assets/robots/unitree_go2/mj_native_electric_actuator.py:14-19, 33-46
#   d->act[i]  = 전류 I  [A]
#   d->ctrl[i] = 제어 입력 (filterexact: 등가전류, user: 전압)
#   force      = gainprm[0] × act = Kt·gr × I
```

```python
# 출처: src/assets/robots/unitree_go2/mj_native_electric_actuator.py:351-353
# ── Gain: force = Kt·gr × I ─────────────────────────
act.gaintype = mujoco.mjtGain.mjGAIN_FIXED
act.gainprm[0] = self._Ktgr
```

운동방정식 자체 (M, C, g, J_c) 의 어셈블리 본문은 mujoco_warp 본체 (`smooth.py`,
`solver.py`, `factor_m`, `solve_m` 등) 에 있고 이 레포에는 vendored
되어 있지 않다 → **8절 미확인**.

> NOTE, hyunnnchoi, 2026.05.02 — 운동방정식과 actuator force 코드의 대응을 명시한다.

> NOTE, hyunnnchoi, 2026.05.02 — 전체 force RHS 에 viscous damping 항 $-b\dot q$ 를 포함한다.

본 절에서는 표준형

$$
M(q)\,\ddot q \;+\; b\dot q \;+\; C(q,\dot q) \;+\; g(q)
\;=\; \tau_{\text{actuator}} \;+\; J_c^{\!\top}\,\lambda,
\qquad \tau_{\text{actuator}} = K_t\,g_r\,I
$$

또는 우변 force 형태로 옮기면

$$
M(q)\,\ddot q
= \tau_{\text{actuator}} - b\dot q - C(q,\dot q) - g(q) + J_c^{\!\top}\lambda
$$

이다.

기호:
- $M(q)$: 관절 위치 $q$ 에서의 mass matrix
- $q$, $\dot q$, $\ddot q$: 관절 위치, 관절속도, 관절가속도
- $b\dot q$: viscous damping 일반화 힘
- $C(q,\dot q)$: Coriolis/centrifugal 항을 포함한 속도 의존 일반화 힘
- $g(q)$: 중력 일반화 힘
- $\tau_{\text{actuator}}$: actuator 가 관절에 가하는 토크
- $J_c$: contact Jacobian, $J_c^\top\lambda$ 는 접촉력이 관절공간으로 투영된 항
- $\lambda$: contact force 또는 constraint impulse 계열의 미지수
- $K_t$: torque constant, $g_r$: gear ratio, $I$: motor 전류

만 가정하고 진행한다. 위 수식의 $\tau_{\text{actuator}} = K_t g_r I$ 는
소스에서 `act.gainprm[0] = self._Ktgr` 로 $K_t g_r$ 를 gain 에 넣고,
MuJoCo 가 `force = gainprm[0] * act` 를 계산하는 방식으로 구현되어 있다.


#### (B) 값 대입 — 자유도 2 축소 모형

> NOTE, hyunnnchoi, 2026.05.02 — 비구속 가속도 수식을 아래 코드 계산과 연결한다.

접촉항을 잠시 빼면 위 운동방정식은 다음 계산으로 축소된다.

> NOTE, hyunnnchoi, 2026.05.02 — 비구속 force RHS 에 $-b\dot q$ 항을 명시한다.

$$
\ddot q_{\mathrm{free}} = M(q)^{-1}\left(\tau_{\mathrm{actuator}} - b\dot q - C(q,\dot q) - g(q)\right)
$$

기호:
- $\ddot q_{\mathrm{free}}$: 접촉 constraint 를 아직 적용하지 않은 비구속 관절가속도
- $M(q)^{-1}$: mass matrix 의 역행렬
- $\tau_{\mathrm{actuator}}$: actuator 토크 벡터
- $b\dot q$: viscous damping 토크 벡터
- $C(q,\dot q)$, $g(q)$: 각각 속도 의존 일반화 힘과 중력 일반화 힘

아래 코드는 이 수식을 `M_inv @ (tau_actuator - damping_qdot - C_qdot - g_q)` 로 구현한다.

In [ ]:
import numpy as np

# 자유도 2 축소 모형 (이 값을 바꿔보세요)
M = np.array([[1.5, 0.1],
              [0.1, 0.8]])              # 질량 행렬 [kg·m²]
q_dot = np.array([0.20, -0.10])          # 관절속도 [rad/s]
damping_b = np.array([0.30, 0.20])       # viscous damping [N·m·s/rad]
# NOTE, hyunnnchoi, 2026.05.02 — force RHS 에서 빠져 있던 b·q̇ damping 항을 뺀다.
damping_qdot = damping_b * q_dot         # b · q̇ [N·m]
C_qdot = np.array([0.05, -0.02])         # Coriolis · q̇ [N·m]
g_q    = np.array([3.00,  0.50])         # 중력 토크   [N·m]

# actuator 토크 (이미 Kt·gr·I 형태로 들어왔다고 가정)
tau_actuator = np.array([4.0, 1.5])      # [N·m]

# 비구속 가속도 (접촉 무시)
M_inv = np.linalg.inv(M)
q_ddot_free = M_inv @ (tau_actuator - damping_qdot - C_qdot - g_q)

print(f"M     =\n{M}")
print(f"M^-1  =\n{M_inv}")
print(f"damping b·q̇ = {damping_qdot}")
print(f"비구속 가속도 q̈_free = {q_ddot_free}")


#### (C) 중간 결과

q̈_free 가 4.2 의 입력. 접촉이 있으면 4.2 에서 λ 가 더해져 q̈ 가 보정된다.


### 4.2 β_imp — Schur complement / Force RHS

#### (A) 소스 발췌 — Schur 항 (qDeriv 누적용) + Force RHS 보정

```python
# 출처: vendor/mujoco_warp_3.6.0_patch/_src/derivative.py:68-89 중 Method A 경로만 발췌
# NOTE, hyunnnchoi, 2026.05.02 — 이 노트는 Method A 전용이므로 dynprm[4]=0 BE 경로만 남긴다.
# Schur term: -(1-β_imp)·Kt·gr·Ke·gr/R for FILTEREXACT motor coupling.
# Method A: 1-β_imp = h/(τ+h)  (BE)
schur = float(0.0)
if actuator_dyntype[actid] == DynType.FILTEREXACT:
    dynprm_act = actuator_dynprm[actuator_dynprm_id, actid]
    Ke_gr = dynprm_act[1]
    L_val = dynprm_act[2]
    if Ke_gr != 0.0 and L_val > MJ_MINVAL:
        tau_e = wp.max(MJ_MINVAL, dynprm_act[0])
        Kt_gr = actuator_gainprm[actuator_gainprm_id, actid][0]
        h_dt = opt_timestep[worldid % opt_timestep.shape[0]]
        one_minus_beta = h_dt / (tau_e + h_dt)
        R_val = L_val / tau_e
        schur = -one_minus_beta * Kt_gr * Ke_gr / R_val
```

```python
# 출처: vendor/mujoco_warp_3.6.0_patch/_src/derivative.py:170, 209-217
# qDeriv 누적과 M_eff = qM − h · qDeriv
qderiv_contrib += moment_i * moment_j * vel
...
qderiv *= opt_timestep[worldid % opt_timestep.shape[0]]
qM_out = qM_in - qderiv         # M_eff = qM − h·qDeriv
```

```python
# 출처: vendor/mujoco_warp_3.6.0_patch/_src/forward.py:752-770 (kernel _actuator_force)
# Method A RHS correction (Schur complement RHS):
#   force currently uses I_old (act_in[act_last]) when not actearly.
#   Method A predicts I_new = β·I_old + (1-β)·ctrl  with  β = 1/(1+h/τ).
#   ΔF = gain·(1-β)·(ctrl - I_old).
if na and act_first >= 0:
    if actuator_dyntype[uid] == DynType.FILTEREXACT:
        dynprm_uid = actuator_dynprm[worldid % actuator_dynprm.shape[0], uid]
        if dynprm_uid[1] != 0.0 and dynprm_uid[2] > MJ_MINVAL and not actuator_actearly[uid]:
            tau_e = wp.max(MJ_MINVAL, dynprm_uid[0])
            h_dt = opt_timestep[worldid % opt_timestep.shape[0]]
            one_minus_beta = h_dt / (tau_e + h_dt)
            I_old = act_in[worldid, act_last]
            force += gain * one_minus_beta * (ctrl - I_old)
```

코드상 명칭 ↔ 본문 기호:
- `one_minus_beta` ↔ $1 - \beta_{\text{imp}}$
- `schur` ↔ Schur 항 $-(1-\beta_{\text{imp}})\,K_t g_r K_e g_r / R$
- `qderiv` (누적, h-스케일 후) ↔ $h \cdot J_c^{\!\top} B J_c$ 의 한 원소
- `force` (after `+= gain · (1−β_imp)·(ctrl−I_old)`) ↔ Force RHS 보정 후 토크

> NOTE, hyunnnchoi, 2026.05.02 — β_imp 관련 수식이 어느 소스 줄에서 구현되는지 풀어 쓴다.

Method A 에서 이 절의 소스가 구현하는 핵심 수식은 다음 두 개다.

> NOTE, hyunnnchoi, 2026.05.02 — 기존 block matrix 표기와 `M_eff` 표기를 함께 적어 Schur 부호를 고정한다.

기존에 쓰던 block matrix 표기로는

$$
\begin{bmatrix}A & B\\ C & D\end{bmatrix}
\begin{bmatrix}\Delta\dot q\\ \Delta i\end{bmatrix}
=
\begin{bmatrix}F\\ G\end{bmatrix}
$$

이고, 전류 증분을 제거하면

$$
\left(A - BD^{-1}C\right)\Delta\dot q = F - BD^{-1}G
$$

이다. 따라서 이 노트에서 말하는 Schur 좌변 보정은
$\mathrm{schur} = -BD^{-1}C$ 이고, RHS 보정은 $-BD^{-1}G$ 이다.
즉 $\Delta\dot q = (A + \mathrm{schur})^{-1}(F + \mathrm{RHS\ correction})$ 로 읽으면 된다.
코드의 `schur = -one_minus_beta * Kt_gr * Ke_gr / R_val` 는 바로 이 음의 부호를 포함한다.

$$
1 - \beta_{\mathrm{imp}} = \frac{h}{\tau_e + h},
\qquad
s = -(1 - \beta_{\mathrm{imp}})\frac{K_t g_r K_e g_r}{R}
$$

$$
F_{\mathrm{post}} = F_{\mathrm{pre}} + K_t g_r(1 - \beta_{\mathrm{imp}})(I_{\mathrm{des}} - I_{\mathrm{old}})
$$

기호:
- $\beta_{\mathrm{imp}}$: Schur complement 와 Force RHS 보정에 쓰는 implicit motor coupling 계수
- $h$: 물리 적분 시간 간격, 여기서는 0.1 ms
- $\tau_e$: 전기 시정수, 코드에서는 $L/R$
- $s$: actuator 하나가 Schur 좌변에 더하는 scalar coupling 항이며 $-BD^{-1}C$ 에 대응
- $K_t g_r$: torque constant 와 gear ratio 의 곱, 전류를 토크로 바꾸는 gain
- $K_e g_r$: back-EMF constant 와 gear ratio 의 곱
- $R$: winding resistance
- $F_{\mathrm{pre}}$: 기존 전류 $I_{\mathrm{old}}$ 만으로 계산된 actuator force/torque
- $F_{\mathrm{post}}$: Schur RHS 보정 후 actuator force/torque
- $I_{\mathrm{des}}$: 목표 전류, $I_{\mathrm{old}}$: 현재 step 시작 시점의 전류

첫 번째 수식은 `one_minus_beta = h_dt / (tau_e + h_dt)` 와
`schur = -one_minus_beta * Kt_gr * Ke_gr / R_val` 로 구현되어 Schur 좌변에 들어간다.
두 번째 수식은 `force += gain * one_minus_beta * (ctrl - I_old)` 로 구현되어
Force RHS 를 보정한다.


#### (B) 값 대입 — 1관절 Schur 1-step + Force RHS

> NOTE, hyunnnchoi, 2026.05.02 — Schur/Force RHS 수식을 값 대입 코드 앞에 둔다.

아래 값 대입 셀은 위 수식을 1 관절, $J_c = 1$ 인 경우로 줄여 계산한다.

> NOTE, hyunnnchoi, 2026.05.02 — `M_eff` 가 $A + \mathrm{schur}$ 역할을 하는 좌변 보정임을 명시한다.
>
> NOTE, hyunnnchoi, 2026.05.02 — contact/λ 풀이는 실제 MuJoCo solver 본문과 다르며, 여기서는 단일 contact 1자유도 임의 축소 예제로만 계산한다.

$$
q\mathrm{Deriv} = h\,J_c^\top s J_c,
\qquad
M_{\mathrm{eff}} = M - q\mathrm{Deriv}
$$

여기서 $s = -BD^{-1}C$ 에 해당하므로 Method A 예제에서는 $s < 0$ 이고,
$q\mathrm{Deriv}<0$ 이 되어 $M_{\mathrm{eff}}=M-q\mathrm{Deriv}$ 가 커진다.
즉 `M_eff` 표기는 기존 block matrix 풀이의 $A + \mathrm{schur}$ 좌변을 mass matrix 쪽 이름으로 쓴 것이다.

$$
\ddot q_{\mathrm{free}} = M_{\mathrm{eff}}^{-1}\left(F_{\mathrm{post}} - b\dot q - C_q - g_q\right)
$$

$$
A = J_c M_{\mathrm{eff}}^{-1}J_c^\top,
\qquad
b = J_c\ddot q_{\mathrm{free}},
\qquad
\lambda = \frac{b}{A}
$$

기호:
- $q\mathrm{Deriv}$: velocity derivative 누적값을 시간 간격 $h$ 로 scale 한 항
- $h$: 물리 적분 시간 간격
- $J_c$: 이 예제에서는 1로 둔 contact/actuator moment Jacobian
- $s$: 위 절의 Schur scalar
- $M_{\mathrm{eff}}$: motor coupling 을 반영한 effective mass
- $A$: 단일 contact Schur complement scalar
- $b$: contact constraint 우변 scalar
- $\lambda$: 단일 contact 에서 풀리는 constraint force/impulse 값
- $\ddot q_{\mathrm{free}}$: 접촉 보정 전 비구속 가속도

코드의 `qderiv`, `M_eff`, `A_schur`, `b_rhs`, `lam` 이 위 항들에 대응한다.

In [ ]:
import numpy as np

# 모터·통합 파라미터 (이 값을 바꿔보세요)
Kt, Ke, gr = 0.128, 0.128, 6.33
R, L       = 0.3, 1e-4
h          = 1e-4              # physics dt = 0.1 ms

tau_e   = L / R                 # 전기 시정수
Kt_gr   = Kt * gr
Ke_gr   = Ke * gr

# β_imp (Method A: dynprm[4]=0 → BE 분기)
one_minus_beta_imp = h / (tau_e + h)
beta_imp           = 1.0 - one_minus_beta_imp
print(f"τ_e        = {tau_e*1e6:.2f} µs")
print(f"β_imp      = {beta_imp:.6f}")
print(f"1 − β_imp  = {one_minus_beta_imp:.6f}")

# Schur scalar (per actuator)
schur = -one_minus_beta_imp * Kt_gr * Ke_gr / R
print(f"schur      = {schur:.6f}")

# 1자유도 단순화: J_c (actuator moment) = 1
J_c_act = 1.0
qderiv_pre = J_c_act * J_c_act * schur     # _qderiv_actuator_passive_actuation_sparse
qderiv     = h * qderiv_pre                # _qderiv_actuator_passive 의 *=h
print(f"qderiv     = {qderiv:.6e}")

# M_eff = M − h·qDeriv (식의 M_eff 자체는 qM − qderiv 로 표기됨, qderiv 가 이미 h 스케일)
M_scalar = 0.5                              # [kg·m²] (이 값을 바꿔보세요)
M_eff    = M_scalar - qderiv                # 양수 qderiv 면 M_eff 가 더 작아지지만,
                                             # Method A 에서는 schur < 0 이므로 qderiv < 0
                                             # → M_eff > M (양정정 강화)
print(f"M = {M_scalar:.6f},  M_eff = {M_eff:.6f}")

# Force RHS 보정
I_old = 1.0                                 # [A]   직전 step 전류
ctrl  = 5.0                                 # [A]   target 전류 (= I_des)
gain  = Kt_gr                               # gainprm[0]
force_pre  = gain * I_old
force_post = force_pre + gain * one_minus_beta_imp * (ctrl - I_old)
print(f"force pre  = {force_pre:.6f} [N·m]")
print(f"force post = {force_post:.6f} [N·m]   (= Schur RHS 보정된 토크)")

# 비구속 가속도 (접촉 무시) — 4.1 의 같은 식을 1자유도로
q_dot_scalar = 0.20                          # [rad/s]
damping_b    = 0.30                          # [N·m·s/rad]
# NOTE, hyunnnchoi, 2026.05.02 — force RHS 에 b·q̇ damping subtraction 을 포함한다.
damping_force = damping_b * q_dot_scalar     # b · q̇ [N·m]
C_q   = 0.0
g_q_v = 2.0                                  # [N·m]
qddot_free = (force_post - damping_force - C_q - g_q_v) / M_eff
print(f"damping b·q̇ = {damping_force:.6f} [N·m]")
print(f"q̈_free    = {qddot_free:.6f}")

# NOTE, hyunnnchoi, 2026.05.02 — contact/λ 계산은 실제 solver 본문이 아니라 임의 1자유도 축소 예제다.
print("주의: contact/λ 풀이는 미확인 solver 본문이 아니라 단일 contact 임의 축소 예제입니다.")
# 단일 접촉의 1자유도 단순 예: J_c = 1 인 normal contact 가정,
# A = J_c · M_eff^-1 · J_cᵀ,  b = J_c · q̈_free  (← 본문 β_imp 의 우변 b)
J_c = 1.0
A_schur = J_c * (1.0 / M_eff) * J_c
b_rhs   = J_c * qddot_free                   # ← Schur complement 우변
lam     = b_rhs / A_schur if A_schur != 0 else 0.0
print(f"A = {A_schur:.6f},  b = {b_rhs:.6f},  λ = {lam:.6f}")

qddot = qddot_free - (1.0 / M_eff) * J_c * lam
print(f"q̈ (corrected) = {qddot:.6f}")


#### (C) 중간 결과

이 절의 핵심은 두 가지 β_imp 사용처를 한 번에 보여준 것이다.

> NOTE, hyunnnchoi, 2026.05.02 — Schur 부호와 damping 을 요약에 반영한다.

1. **좌변 (M_eff):** `qDeriv += −(1−β_imp)·Kt·gr·Ke·gr/R · J_cᵀJ_c` →
   `M_eff = qM − h·qDeriv` (`derivative.py`). 여기서 Schur 는 기존 표기의 $-BD^{-1}C$ 이므로 음의 부호를 포함한다.
2. **우변 (force):** actuator 쪽 RHS 는 `force += gain·(1−β_imp)·(ctrl − I_old)` (`forward.py`) 로 보정되고,
   전체 운동방정식 RHS 에서는 추가로 `−b·q_dot − C_q − g_q` 를 뺀다.

좌·우변 모두 같은 $1-\beta_{\mathrm{imp}} = h/(\tau+h)$ 를 사용하는 것이 Method A 의 정의이고
(`dynprm[4] = 0`), 그 결과 풀린 $\ddot q$ 가 4.3 의 입력이 된다.

기호: 여기서 $\tau$ 는 위 절의 전기 시정수 $\tau_e$ 와 같은 역할이고,
$\ddot q$ 는 접촉 보정까지 끝난 관절가속도다.


### 4.3 β_int — 적분 단계

#### (A) 소스 발췌 — β_int 적분기 + act_dot

```python
# 출처: vendor/mujoco_warp_3.6.0_patch/_src/forward.py:147-170 중 Method A 경로만 발췌
# NOTE, hyunnnchoi, 2026.05.02 — 이 노트는 Method A 전용이므로 dynprm[4]=0 BE 적분만 남긴다.
if actuator_dyntype == DynType.FILTEREXACT:
    tau = wp.max(MJ_MINVAL, actuator_dynprm[0])
    # Motor coupling detection: dynprm[1]=Ke*gr != 0 AND dynprm[2]=L > 0
    # Method A: β_int = 1/(1+h/τ)  (BE)
    if actuator_dynprm[1] != 0.0 and actuator_dynprm[2] > MJ_MINVAL:
        act = act_in + act_dot_scale * act_dot_in * opt_timestep / (1.0 + opt_timestep / tau)
```

```python
# 출처: vendor/mujoco_warp_3.6.0_patch/_src/forward.py:679-693 (kernel _actuator_force, FILTEREXACT 분기)
elif dyntype == DynType.FILTEREXACT:
    # Coupled filterexact: standard filter + Ke mismatch correction.
    #   dI/dt = (ctrl - act) / tau  +  (Ke_nom*gr - Ke_plant*gr) * omega / L
    act = act_in[worldid, act_last]
    tau_e = wp.max(MJ_MINVAL, dynprm[0])
    act_dot = (ctrl - act) / tau_e
    L_dyn = dynprm[2]
    if L_dyn > MJ_MINVAL:
        # omega at step start; ZOH over dt (kernel picks latest actuator_velocity).
        omega = actuator_velocity_in[worldid, uid]
        act_dot += (dynprm[3] - dynprm[1]) * omega / L_dyn
```

코드상 명칭 ↔ 본문 기호:
- `act_in`, `act_out` (또는 갱신된 `act`) ↔ $I_n$, $I_{n+1}$
- `act_dot_in` (= 위에서 계산된 `act_dot`) ↔ $\dot I$
- `opt_timestep / (1 + opt_timestep / tau)` ↔ Method A 적분 식의
  $h \cdot \dot I / (1 + h/\tau) = (1 - \beta_{\text{int}})\,\tau\,\dot I$
  ($\beta_{\text{int}} = 1/(1+h/\tau)$).

> NOTE, hyunnnchoi, 2026.05.02 — `filterexact` 전류 필터 수식과 코드 구현을 연결한다.

`filterexact` 분기에서 전류 미분은 다음 수식으로 계산된다.

$$
\dot I = \frac{I_{\mathrm{des}} - I_n}{\tau_e}
+ \frac{(K_{e,\mathrm{nom}}g_r - K_{e,\mathrm{plant}}g_r)\omega}{L}
$$

Method A 의 전류 적분은 다음 수식이다.

$$
I_{n+1} = I_n + \frac{h\dot I}{1 + h/\tau_e},
\qquad
\beta_{\mathrm{int}} = \frac{1}{1 + h/\tau_e}
$$

$K_{e,\mathrm{nom}} = K_{e,\mathrm{plant}}$ 이면 mismatch 항이 0 이라
$I_{n+1}=\beta_{\mathrm{int}}I_n+(1-\beta_{\mathrm{int}})I_{\mathrm{des}}$ 이고,
두 값이 다르면 Method A 에서 다음 추가항이 붙는다.

$$
I_{n+1}=\beta_{\mathrm{int}}I_n+(1-\beta_{\mathrm{int}})I_{\mathrm{des}}
+ \frac{h}{1+h/\tau_e}\frac{(K_{e,\mathrm{nom}}g_r-K_{e,\mathrm{plant}}g_r)\omega}{L}
$$

기호:
- $I_n$, $I_{n+1}$: 현재 substep 시작/끝의 motor 전류
- $\dot I$: 전류 시간미분, 코드의 `act_dot`
- $I_{\mathrm{des}}$: PD 단계에서 계산되어 `ctrl` 로 전달되는 목표 전류
- $\tau_e$: 전기 시정수, $L/R$
- $L$: winding inductance
- $K_{e,\mathrm{nom}}g_r$: actuator 설정에 들어간 nominal back-EMF gain
- $K_{e,\mathrm{plant}}g_r$: plant 쪽 back-EMF gain
- $\omega$: actuator/joint velocity
- $\beta_{\mathrm{int}}$: Method A 전류 적분에 쓰는 implicit filter 계수
- $h$: 물리 적분 시간 간격

첫 번째 수식은 `_actuator_force` 안의
`act_dot = (ctrl - act) / tau_e` 와 mismatch 보정 `act_dot += ... * omega / L_dyn` 로
구현된다. 두 번째 수식은 `act = act_in + ... * opt_timestep / (1.0 + opt_timestep / tau)` 로 구현된다.

기계측 적분 (semi-implicit Euler: q̇ ← q̇ + h·q̈, q ← q + h·q̇_new) 본문은
mujoco_warp `_src/euler.py` 외부 → **8절 미확인**. 본 셀에서는 표준형을
직접 적용한다.


#### (B) 값 대입 — 1관절 적분

> NOTE, hyunnnchoi, 2026.05.02 — 기계측/전기측 적분 수식을 코드 앞에 둔다.
>
> NOTE, hyunnnchoi, 2026.05.02 — 기계측 적분 본문은 미확인이라 아래 semi-implicit Euler 는 표준형을 임의 적용한 예제다.

아래 코드는 기계측 semi-implicit Euler 와 전기측 Method A 적분을 각각 구현한다.

$$
\dot q_{n+1} = \dot q_n + h\ddot q_n,
\qquad
q_{n+1} = q_n + h\dot q_{n+1}
$$

$$
I_{n+1} = I_n + \frac{h}{1+h/\tau_e}\frac{I_{\mathrm{des}}-I_n}{\tau_e}
$$

기호:
- $q_n$, $q_{n+1}$: 현재/다음 substep 의 관절 위치
- $\dot q_n$, $\dot q_{n+1}$: 현재/다음 substep 의 관절속도
- $\ddot q_n$: 현재 substep 에서 dynamics 로 계산한 관절가속도
- $I_n$, $I_{n+1}$: 현재/다음 substep 의 motor 전류
- $I_{\mathrm{des}}$: PD 단계에서 전달된 목표 전류
- $h$: 물리 적분 시간 간격
- $\tau_e$: 전기 시정수

코드의 `q_dot_new`, `q_new`, `I_np1` 가 위 세 결과값이다.

In [ ]:
import numpy as np

# 4.2 에서 받은 q, q̇, q̈
q     = np.array([0.05])
q_dot = np.array([0.10])
q_ddot = np.array([qddot])    # 4.2 (B) 의 마지막 출력

dt = 1e-4    # = physics timestep   (이 값을 바꿔보세요)

# NOTE, hyunnnchoi, 2026.05.02 — 기계측 적분 본문은 미확인이라 표준 semi-implicit Euler 를 임의 적용한다.
print("주의: 기계측 적분은 미확인 본문 대신 표준 semi-implicit Euler 를 적용한 예제입니다.")
# 기계측 semi-implicit Euler
q_dot_new = q_dot + dt * q_ddot
q_new     = q     + dt * q_dot_new
print(f"q̇_new = {q_dot_new}")
print(f"q_new  = {q_new}")

# 전기측: β_int (Method A) 전류 적분
Kt, Ke, gr = 0.128, 0.128, 6.33
R,  L      = 0.3, 1e-4
tau_e      = L / R

# act_dot 본문 (healthy: dynprm[3] = dynprm[1] → Ke mismatch 항 = 0)
I_n        = 1.0
ctrl_amps  = 5.0
act_dot    = (ctrl_amps - I_n) / tau_e

# Method A: I_{n+1} = I_n + h · act_dot / (1 + h/τ)
I_np1      = I_n + dt * act_dot / (1.0 + dt / tau_e)

# 등가 형식: I_{n+1} = β·I_n + (1−β)·ctrl,  β = 1/(1+h/τ)
beta_int   = 1.0 / (1.0 + dt / tau_e)
I_np1_eq   = beta_int * I_n + (1.0 - beta_int) * ctrl_amps

print(f"β_int      = {beta_int:.6f}")
print(f"I_{{n+1}}      = {I_np1:.6f} [A]")
print(f"I_{{n+1}} (β형) = {I_np1_eq:.6f} [A]   ← 두 형식이 일치해야 함")


#### (C) 중간 결과

한 substep (0.1 ms) 후의 상태 = `(q_new, q_dot_new, I_np1)`.
다음 substep 의 4.1 입력으로 들어간다.


### 4.4 한 substep 요약

> NOTE, hyunnnchoi, 2026.05.02 — 한 substep 의 수식 흐름을 아래 통합 코드와 연결한다.

화살표 한 줄:
$\tau_{\text{des}}, q, \dot q, I \to$ **운동방정식** $\to$
**β_imp (Schur · Force RHS)** $\to$ **β_int (적분)** $\to$
다음 $(q, \dot q, I)$.

아래 셀은 4.1 ∼ 4.3 의 수식

$$
M_{\mathrm{eff}} = M - hJ_c^\top sJ_c,
\qquad
F_{\mathrm{post}} = K_t g_r I + K_t g_r(1-\beta_{\mathrm{imp}})(I_{\mathrm{des}}-I),
\qquad
\ddot q = M_{\mathrm{eff}}^{-1}(F_{\mathrm{post}} - b\dot q - C_q - g_q),
\qquad
I_{n+1}=I_n+\frac{h\dot I}{1+h/\tau_e}
$$

> NOTE, hyunnnchoi, 2026.05.02 — 통합 substep 식에도 damping subtraction 을 포함한다.

기호:
- $M_{\mathrm{eff}}$: 한 substep dynamics 풀이에 쓰는 effective mass
- $M$: 원래 mass scalar 또는 mass matrix
- $J_c$: 이 통합 예제에서 1로 둔 contact/actuator Jacobian
- $s$: motor coupling Schur scalar, 기존 표기의 $-BD^{-1}C$
- $F_{\mathrm{post}}$: Schur RHS 보정 후 actuator torque
- $b\dot q$: viscous damping 토크
- $I$: 현재 전류, $I_{\mathrm{des}}$: 목표 전류
- $\beta_{\mathrm{imp}}$: Schur/Force RHS 쪽 implicit 계수
- $I_n$, $I_{n+1}$: 전류 적분 전/후 값
- $\dot I$: 전류 시간미분
- $h$: 물리 적분 시간 간격, $\tau_e$: 전기 시정수

을 `one_substep(...)` 함수로 묶어 1 PD 주기 (5 ms = 50 substep) 동안 반복하고,
(q, q̇, I) 변화를 표로 출력한다.


In [ ]:
import numpy as np
import pandas as pd

def one_substep(q, q_dot, I, ctrl, M, damping_b, C_q, g_q_v, params):
    # 1 자유도 축소 모형의 한 substep (4.1 → 4.2 → 4.3).
    Kt, Ke, gr = params['Kt'], params['Ke'], params['gr']
    R,  L,  h  = params['R'],  params['L'],  params['h']
    Kt_gr      = Kt * gr
    Ke_gr      = Ke * gr
    tau_e      = L / R

    # ── 4.2 β_imp ──────────────────────────────────────────
    omb        = h / (tau_e + h)              # 1 − β_imp
    schur      = -omb * Kt_gr * Ke_gr / R
    qderiv     = h * (1.0 * 1.0 * schur)      # J=1
    M_eff      = M - qderiv                    # M_eff = M − h·qDeriv

    gain       = Kt_gr
    force      = gain * I + gain * omb * (ctrl - I)   # Schur RHS 보정 후 actuator torque

    # ── 4.1 비구속 가속도 (접촉 무시) ───────────────────────
    # NOTE, hyunnnchoi, 2026.05.02 — 전체 force RHS 에 b·q̇ damping subtraction 을 포함한다.
    damping_force = damping_b * q_dot
    qddot      = (force - damping_force - C_q - g_q_v) / M_eff

    # ── 4.3 적분 ────────────────────────────────────────────
    q_dot_new  = q_dot + h * qddot
    q_new      = q     + h * q_dot_new

    act_dot    = (ctrl - I) / tau_e            # healthy
    I_new      = I + h * act_dot / (1.0 + h / tau_e)

    return q_new, q_dot_new, I_new


# 파라미터 (이 값을 바꿔보세요)
params = dict(Kt=0.128, Ke=0.128, gr=6.33, R=0.3, L=1e-4, h=1e-4)

# NOTE, hyunnnchoi, 2026.05.02 — PD 단계에서 계산한 I_des 를 50 substep 동안 같은 값으로 사용한다.
ctrl = 5.0      # = I_des [A]

# 단순화한 1자유도 동역학 (이 값을 바꿔보세요)
# NOTE, hyunnnchoi, 2026.05.02 — substep 예제 파라미터에 viscous damping b 를 추가한다.
M, damping_b, C_q, g_q_v = 0.5, 0.30, 0.0, 0.0

# 초기 상태
q, q_dot, I = 0.0, 0.0, 0.0

N_SUBSTEPS_PER_PD = 50    # 5 ms / 0.1 ms (이 값을 바꿔보세요)

rows = [{"k": 0, "q": q, "q_dot": q_dot, "I": I}]
for k in range(1, N_SUBSTEPS_PER_PD + 1):
    q, q_dot, I = one_substep(q, q_dot, I, ctrl, M, damping_b, C_q, g_q_v, params)
    rows.append({"k": k, "q": q, "q_dot": q_dot, "I": I})

df = pd.DataFrame(rows)
print(df.to_string(index=False, float_format=lambda x: f"{x: .6f}"))


## 5. 전체 흐름도

> NOTE, hyunnnchoi, 2026.05.02 — 흐름도 안의 축약 수식 기호를 아래에 정리한다.

```mermaid
flowchart TB
    subgraph POL["정책 (20 ms)"]
        OBS["관측: v_body, ω_body, g_proj, v_cmd, q, q̇, q_des_prev"]
        OBS -- "신경망" --> QDES["q_des"]
    end
    subgraph PDC["PD (5 ms × 4)"]
        QDES --> PDF["τ_des = clip( Kp·(q_des − q) − Kd·q̇,  ±τ_max(ω) )"]
        PDF  --> IDS["I_des = τ_des / (Kt·gr)"]
    end
    subgraph PHY["물리 (0.1 ms × 50 per PD)"]
        IDS  --> EOM["M(q)·q̈ + b·q̇ + C(q,q̇) + g(q) = Kt·gr·I + J_cᵀλ"]
        EOM  --> SCH["β_imp · Schur:  qDeriv += −(1−β_imp)·Kt·gr·Ke·gr/R · J_cᵀJ_c<br/>force += gain·(1−β_imp)·(ctrl − I_old)"]
        SCH  --> INT["β_int · 적분:  I_{n+1} = I_n + h·act_dot / (1 + h/τ)<br/>q̇_{n+1} = q̇_n + h·q̈,  q_{n+1} = q_n + h·q̇_{n+1}"]
        INT  --> EOM
    end
    PHY -- "(q, q̇, I) 마지막 substep" --> POL
```

기호:
- $q_{\mathrm{des}}$: 정책이 출력한 목표 관절각
- $\tau_{\mathrm{des}}$: PD와 motor saturation 후 목표 토크
- $I_{\mathrm{des}}$: 목표 전류
- $M(q)$, $C(q,\dot q)$, $g(q)$: mass matrix, 속도 의존 일반화 힘, 중력 일반화 힘
- $J_c^\top\lambda$: 접촉 constraint force 를 관절공간으로 투영한 항
- $\beta_{\mathrm{imp}}$: Schur/Force RHS 보정 계수
- $\beta_{\mathrm{int}}$: 전류 적분 계수
- $h$: 물리 적분 시간 간격


## 6. 한 정책 주기 누적 호출 횟수

| 단계 | 주기 | 정책 한 주기당 호출 횟수 |
|------|------|--------------------------|
| 정책 | 20 ms | 1 |
| PD | 5 ms | 4 (= 20 ms / 5 ms) |
| 물리 | 0.1 ms | 200 (= 20 ms / 0.1 ms = `decimation`) |

PD 1 회당 물리 substep 수 = 50 (= 5 ms / 0.1 ms = `pd_substeps`).


## 7. 구현 위치 참조표

> NOTE, hyunnnchoi, 2026.05.02 — 값 유지 항목을 수식 형태로 표기한다.

> NOTE, hyunnnchoi, 2026.05.02 — 8절 미확인 항목 중 mujoco engine 본체에 있는 것들의
> 위치를 추가한다. 출처 prefix `mujoco fork:` 는 사용자 fork
> `https://github.com/sunyoung-1206/mujoco` 의 같은 경로를 가리킨다.
> 컬럼의 "본문 기호" 는 노트의 식·기호와의 대응을 보여주기 위한 것이다.
>
> 기호:
> - $h$ : 물리 적분 시간 간격 (= `m->opt.timestep`)
> - $M$ : mass matrix (`d->qM`, factor 후 `d->qLD`)
> - $C(q,\dot q),\,g(q)$ : Coriolis·중력 일반화 힘. mujoco 에서는 이 둘이
>   `qfrc_bias` 한 벡터로 합쳐서 RNE 로 계산된다.
> - $J_c$ : 접촉/등식·부등식 constraint Jacobian (`d->efc_J`,
>   행 = `nefc`, 열 = `nv`)
> - $\lambda$ : constraint force/impulse (`d->efc_force`, 관절공간 투영은 `d->qfrc_constraint`)
> - $\ddot q_{\mathrm{free}}$ : 비구속 가속도 (`d->qacc_smooth`)
>   $= M^{-1}(\,$`qfrc_passive` $-$ `qfrc_bias` $+$ `qfrc_applied` $+$ `qfrc_actuator`$)$

| 단계 | 본문 기호 | 코드상 명칭 | 파일 | 라인 |
|------|----------|-----------|------|------|
| 정책 | task 등록 | `register_mjlab_task("Unitree-Go2-Flat-MethodA-Electric", ...)` | `src/tasks/velocity/config/go2/__init__.py` | 81-93 |
| 정책 | 시간 설정 | `cfg.sim.mujoco.timestep`, `cfg.decimation` | `src/tasks/velocity/config/go2/env_cfgs.py` | 206-207 |
| PD | substep 상수 | `_COUPLED_SUBSTEPS = 200`, `_PD_RECOMPUTE = 50` | `src/assets/robots/unitree_go2/go2_constants.py` | 305-306 |
| PD | actuator cfg | `_MA_MOTOR`, `GO2_METHODA_HIP/THIGH/CALF` | `src/assets/robots/unitree_go2/go2_constants.py` | 342-358 |
| PD | $\tau_{\mathrm{des}}$ | `tau_des = super().compute(cmd)` | `src/assets/robots/unitree_go2/mj_native_electric_actuator.py` | 509 |
| PD | $I_{\mathrm{des}}$ | `I_des = tau_des / self._Ktgr` | `src/assets/robots/unitree_go2/mj_native_electric_actuator.py` | 531 |
| PD | $I_{\mathrm{des}}(t)=I_{\mathrm{des}}(t_m)$ 값 유지 | `_I_des_hold`, `pd_substeps` | `src/assets/robots/unitree_go2/mj_native_electric_actuator.py` | 497-540 |
| 물리 | dynprm[4] = method | `act.dynprm[4] = _METHOD_TO_DYNPRM4[cfg.method]` | `src/assets/robots/unitree_go2/mj_native_electric_actuator.py` | 333 |
| 물리 | $1-\beta_{\mathrm{imp}}$ (Schur, A 분기) | `one_minus_beta = h_dt / (tau_e + h_dt)` | `vendor/mujoco_warp_3.6.0_patch/_src/derivative.py` | 87 |
| 물리 | Schur scalar $s$ | `schur = -one_minus_beta * Kt_gr * Ke_gr / R_val` | `vendor/mujoco_warp_3.6.0_patch/_src/derivative.py` | 89 |
| 물리 | qDeriv 누적 ($J_c^\top s J_c$) | `qderiv_contrib += moment_i * moment_j * vel` | `vendor/mujoco_warp_3.6.0_patch/_src/derivative.py` | 170 |
| 물리 | $M_{\mathrm{eff}} = qM - h \cdot$ qDeriv | `qM_in - qderiv` (after `qderiv *= h`) | `vendor/mujoco_warp_3.6.0_patch/_src/derivative.py` | 209-217 |
| 물리 | $\beta_{\mathrm{imp}}$ Force RHS | `force += gain * one_minus_beta * (ctrl - I_old)` | `vendor/mujoco_warp_3.6.0_patch/_src/forward.py` | 752-770 |
| 물리 | $\dot I$ (filterexact) | `act_dot = (ctrl - act) / tau_e + (dynprm[3]-dynprm[1])*omega/L` | `vendor/mujoco_warp_3.6.0_patch/_src/forward.py` | 679-693 |
| 물리 | $\beta_{\mathrm{int}}$ (Method A) | `act = act_in + ... * h / (1 + h/τ)` | `vendor/mujoco_warp_3.6.0_patch/_src/forward.py` | 163-164 |
| 물리 | force = $K_t g_r I$ | `act.gainprm[0] = self._Ktgr` | `src/assets/robots/unitree_go2/mj_native_electric_actuator.py` | 351-353 |
| 물리 | $M$ 어셈블리 (CRB) | `mj_crb(m, d)` → `d->qM` | mujoco fork: `src/engine/engine_core_smooth.c` | 1745 |
| 물리 | $M$ factorization (LᵀDL) | `mj_factorM` → `mj_factorI` | mujoco fork: `src/engine/engine_core_smooth.c` | 1894 / 1924 |
| 물리 | $M^{-1} \cdot (\,\cdot\,)$ 풀이 | `mj_solveM`, `mj_solveLD` | mujoco fork: `src/engine/engine_core_smooth.c` | 2152 / 2064 |
| 물리 | qfrc_bias $= C(q,\dot q) + g(q)$ (RNE) | `mj_rne(m, d, /*flg_acc=*/0, qfrc_bias)` | mujoco fork: `src/engine/engine_core_smooth.c` | 2359 |
| 물리 | qfrc_smooth, $\ddot q_{\mathrm{free}}$ 어셈블리 | `mj_fwdAcceleration` (`qacc_smooth = M \\ qfrc_smooth`) | mujoco fork: `src/engine/engine_forward.c` | 788-821 |
| 물리 | $J_c$ 어셈블리 (`efc_J`) | `mj_makeConstraint` | mujoco fork: `src/engine/engine_core_constraint.c` | 2495 |
| 물리 | $J_c \cdot v$ 곱셈 | `mj_mulJacVec`, `mj_mulJacTVec` | mujoco fork: `src/engine/engine_core_constraint.c` | 576 / 595 |
| 물리 | 접촉력 $\lambda$ 풀이 (디스패치) | `mj_fwdConstraint` | mujoco fork: `src/engine/engine_forward.c` | 953 |
| 물리 | $\lambda$ 풀이 본문 (CG / Newton) | `mj_solCG_island`, `mj_solNewton_island` | mujoco fork: `src/engine/engine_solver.c` | (각 함수 참조) |
| 물리 | semi-implicit Euler $\dot q \mathrel{+}= h\ddot q$, $q \mathrel{+}= h\dot q$ | `mj_advance` (`mju_addToScl(qvel, qacc, h, nv)`, `mj_integratePosInd`) | mujoco fork: `src/engine/engine_forward.c` | 1047-1154 (1126, 1132) |
| 물리 | mj_Euler 디스패치 (damping 분기 포함) | `mj_Euler` → `mj_EulerSkip` | mujoco fork: `src/engine/engine_forward.c` | 1156-1241 |


## 8. 미확인 항목

> NOTE, hyunnnchoi, 2026.05.02 — 1차 작성 시 "미확인" 으로 둔 6 항목 중 5 항목을
> mujoco fork (`https://github.com/sunyoung-1206/mujoco`) 에서 위치 확인. 7절
> 참조표 마지막 8 행이 그 결과다. 본 절에는 (1) 새로 확인된 항목의 mujoco engine
> 안에서의 위치 매핑을 짧게 기록하고, (2) 여전히 코드 본문 확인 못 한 항목만
> "미확인" 으로 남긴다.
>
> 노트북 (B) 값 대입 셀들의 1∼2 자유도 축소 모형은 그대로 유지한다 — 이는 흐름과
> 단위 정합 확인용이며, 실제 mujoco 의 sparse LᵀDL / Newton iteration 본문을 옮긴
> 것이 아니다.

### 8.1 mujoco fork 에서 위치 확인된 항목 (이전 미확인 → 확인됨)

기호 정의:
- $\ddot q_{\mathrm{free}}$ : 비구속 가속도 (`d->qacc_smooth`)
- $M$ : mass matrix (`d->qM`), factor 후 (`d->qLD`)
- $C(q,\dot q) + g(q)$ : Coriolis·중력 합 일반화 힘. mujoco 에서는 `qfrc_bias`
  한 벡터로 합쳐 RNE 로 계산
- $J_c$ : constraint Jacobian (`d->efc_J`, 행 = `nefc`, 열 = `nv`)
- $\lambda$ : constraint force (`d->efc_force`); 관절공간 투영 = `d->qfrc_constraint`
- $h$ : 물리 적분 시간 간격

| 본문 항목 | mujoco fork 위치 | 핵심 식 / 호출 |
|----------|-----------------|---------------|
| 4.1 운동방정식의 비구속 가속도 $\ddot q_{\mathrm{free}}$ 어셈블리 | `src/engine/engine_forward.c:788-821` (`mj_fwdAcceleration`) | (i) `qfrc_smooth = qfrc_passive − qfrc_bias + qfrc_applied + qfrc_actuator`, (ii) `qacc_smooth = M⁻¹·qfrc_smooth` (= `mj_solveLD`) |
| 4.1 의 $M$ 어셈블리 (Composite Rigid Body) | `src/engine/engine_core_smooth.c:1745` (`mj_crb`) | `d->qM` 채움 |
| 4.1 의 $M$ factorization | `src/engine/engine_core_smooth.c:1894` (`mj_factorM`) → `:1924` (`mj_factorI`) | sparse LᵀDL: `d->qLD`, `d->qLDiagInv` |
| 4.1 의 $M^{-1}\cdot(\cdot)$ 풀이 | `src/engine/engine_core_smooth.c:2152` (`mj_solveM`), `:2064` (`mj_solveLD`) | factor 결과로 backsubstitution |
| 4.1 의 $C(q,\dot q) + g(q)$ (= `qfrc_bias`) 계산 | `src/engine/engine_core_smooth.c:2359` (`mj_rne`) | Recursive Newton-Euler. 호출 시 `flg_acc = 0` 이면 결과는 Coriolis + 중력 일반화 힘만 |
| 4.2 의 constraint Jacobian $J_c$ (= `efc_J`) 어셈블리 | `src/engine/engine_core_constraint.c:2495` (`mj_makeConstraint`) | $J_c$ 의 $v$ 곱은 `mj_mulJacVec` (`:576`), $J_c^\top$ 곱은 `mj_mulJacTVec` (`:595`) |
| 4.2 의 접촉력 $\lambda$ Schur 풀이 (디스패치) | `src/engine/engine_forward.c:953` (`mj_fwdConstraint`) | 옵션에 따라 `mj_solCG_island` / `mj_solNewton_island` 로 분기 |
| 4.2 의 $\lambda$ 풀이 본문 (CG / Newton iteration) | `src/engine/engine_solver.c` 안 `mj_solCG_island`, `mj_solNewton_island` | 1자유도 축소 예제의 $\lambda = b/A$ 와는 달리 실제로는 nefc 차원 KKT/dual 문제를 iterative 로 푼다 |
| 4.3 의 semi-implicit Euler 본문 | `src/engine/engine_forward.c:1156-1241` (`mj_EulerSkip` / `mj_Euler`) | damping 없으면 `qacc` 그대로, 있으면 `qH = M + h·diag(B)` factor 후 `M_eff⁻¹·qfrc` |
| 4.3 의 $\dot q \mathrel{+}= h\,\ddot q$, $q \mathrel{+}= h\,\dot q$ 본문 | `src/engine/engine_forward.c:1047-1154` (`mj_advance`) | `:1126` `mju_addToScl(d->qvel, qacc, h, nv)` (속도 갱신), `:1132` `mj_integratePosInd(...)` (위치 갱신, qvel 인자에 갱신된 속도가 들어가서 semi-implicit 됨) |

> NOTE, hyunnnchoi, 2026.05.02 — 노트북 4.3 의 (B) 셀에서
> `q_dot_new = q_dot + dt * q_ddot` 다음에
> `q_new = q + dt * q_dot_new` 를 쓴 것이 위 `mj_advance` 의
> "advance velocities → advance positions with `qvel` (이미 갱신된 값) "
> 순서와 정확히 같다 (= semi-implicit / symplectic Euler).

> NOTE, hyunnnchoi, 2026.05.02 — `mj_Euler` 의 damping 분기는 노트북에 반영하지
> 않았다. mjlab 의 Method A actuator 는 implicit damping 을 mujoco_warp 의 patched
> `derivative.py` 에서 qDeriv 형태로 처리하므로 (`M_eff = qM − h·qDeriv`),
> mujoco 의 `mj_Euler` damping 경로 (`qH = M + h·diag(B)`) 와는 별도 경로다.
> 단, 본 노트북은 이 patched 경로만 다루므로 `mj_EulerSkip:1191-1227` 의 damping
> 분기 본문은 더 들여다보지 않아도 된다.

### 8.2 여전히 미확인

| 항목 | 사유 |
|------|------|
| `DcMotorActuator.compute()` 본문 (PD 식 + DC motor velocity-dependent saturation) | 외부 패키지 `mjlab.actuator.dc_actuator` 안. mujoco fork 에는 없음 (mjlab 은 별도 repo). 노트북 3 절의 PD 식은 cfg 값과 wrapper docstring 으로부터 재구성한 표준형 $\tau_{pd} = K_p (q_{des}-q) - K_d \dot q$. saturation 도 표준형 $\tau_{\max}(\omega) = \mathrm{sat}\cdot \max(0, 1-|\omega|/v_{\max})$ 가정. |
| 위 saturation 의 정확한 함수 형태 | 동일 — mjlab 본문 미확인. linear cutoff / quadratic / per-quadrant 형태 등 변형이 있을 수 있다. |

### 8.3 (B) 값 대입 셀의 단순화 표시

- 4.2 의 $\lambda$ 풀이는 1 자유도, $J_c=1$ 의 단일 contact 단순화
  ($A = J_c M_{\mathrm{eff}}^{-1} J_c^\top$, $b = J_c \ddot q_{\mathrm{free}}$,
  $\lambda = b/A$). 실제 mujoco 는 위 §8.1 표대로 nefc 차원의 KKT/dual 문제를
  Newton 또는 CG 로 푼다.
- 4.1 / 4.4 의 2 자유도 또는 1 자유도 $M, C, g$ 는 직접 입력값. 실제로는 `mj_crb`
  + `mj_rne` 가 12 자유도 트리에서 자동 계산한다.
- 4.3 의 semi-implicit Euler 는 mujoco 의 `mj_advance` 와 동일한 순서로
  옮겨 적었다 (속도 먼저, 그 다음 위치).
